#### Modules

In [5]:
import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

#### Original CNN Architecture (putting the parameters back in)

In [ ]:
# Keep architecture identical to the training notebook so state_dict loading matches.
class CNNBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 30 * 30, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),)
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

#### Utilize GPU

In [3]:
# Use GPU when available.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

#### Reload models and metadata from phase 2

In [4]:
# Define paths to the model files and metadata
root = Path(".")
walls_model_path = root / "model_walls.pth"
decks_model_path = root / "model_decks.pth"
metadata_path = root / "model_metadata.json"

# Use GPU for CNN
model_walls = CNNBinary().to(device)
model_decks = CNNBinary().to(device)

# Load the parameters of the trained model (state_dict) into the model architecture
model_walls.load_state_dict(torch.load(walls_model_path, map_location=device))
model_decks.load_state_dict(torch.load(decks_model_path, map_location=device))

# Load the thresholds from the metadata file
model_walls.eval()
model_decks.eval()
with metadata_path.open("r", encoding="utf-8") as f:
    metadata = json.load(f)
best_threshold_walls = metadata.get("best_threshold_walls", 0.5)
best_threshold_decks = metadata.get("best_threshold_decks", 0.5)

# Check whether the models and metadata are loaded correctly
print("Models and metadata loaded successfully.")
print(f"Device: {device}")
print(f"Walls threshold: {best_threshold_walls}")
print(f"Decks threshold: {best_threshold_decks}")

Models and metadata loaded successfully.
Device: cuda
Walls threshold: 0.45999999999999985
Decks threshold: 0.48999999999999977


#### Get 10 sample images from the original dataset

In [ ]:
# Load sample images from the original datasets
df_walls = pd.read_pickle('wall_data.pkl')
df_decks = pd.read_pickle('deck_data.pkl')

# Select 5 cracked and 5 non-cracked wall images for visualization
cracked_walls = df_walls[df_walls['Label'] == 'Cracked'].sample(5, random_state=42)
non_cracked_walls = df_walls[df_walls['Label'] == 'Non-Cracked'].sample(5, random_state=42)

# Combine them
sample_images = pd.concat([cracked_walls, non_cracked_walls]).reset_index(drop=True)

print(f"Selected {len(sample_images)} sample images for filter visualization")
print(f"Distribution: {sample_images['Label'].value_counts().to_dict()}")

Selected 10 sample images for filter visualization
Distribution: {'Cracked': 5, 'Non-Cracked': 5}


#### Extract Learned Filters from First Convolutional Layer

In [ ]:
# Function to apply convolutional filters defined in the CNN model to an image and return the feature maps.
def apply_conv_layer(image, model, layer_idx=0):
    
    # Prepare image tensor
    img_tensor = torch.from_numpy(image.astype(np.float32) / 255.0)
    
    # Ensure it's 3 channels
    if img_tensor.ndim == 2:
        img_tensor = img_tensor.unsqueeze(0).repeat(3, 1, 1)
    elif img_tensor.shape[2] == 3:
        img_tensor = img_tensor.permute(2, 0, 1)  # HWC to CHW
    
    img_tensor = img_tensor.unsqueeze(0).to(device)  # Add batch dimension
    
    # Apply first conv layer and ReLU
    with torch.no_grad():
        conv_output = model.features[0](img_tensor)  # Conv2d
        activated_output = model.features[1](conv_output)  # ReLU
    
    return activated_output.squeeze(0).cpu().numpy()

In [ ]:
# Visualize original images and selected filter responses for 10 sample images
num_filters_to_show = 8  # Show 8 most representative filters
filter_indices = [0, 4, 8, 12, 16, 20, 24, 28]  # Sample evenly across 32 filters

for img_idx in range(len(sample_images)):
    row = sample_images.iloc[img_idx]
    image = row['ImageData']
    label = row['Label']
    
    # Apply filters
    feature_maps = apply_conv_layer(image, model_walls)
    
    # Create visualization
    fig = plt.figure(figsize=(18, 3))
    gs = GridSpec(1, num_filters_to_show + 1, figure=fig)
    
    # Original image
    ax0 = fig.add_subplot(gs[0, 0])
    if image.ndim == 2:
        ax0.imshow(image, cmap='gray')
    else:
        ax0.imshow(image)
    ax0.set_title(f'Original\\n{label}', fontsize=10, fontweight='bold')
    ax0.axis('off')
    
    # Filter responses
    for i, filter_idx in enumerate(filter_indices):
        ax = fig.add_subplot(gs[0, i + 1])
        feature_map = feature_maps[filter_idx]
        
        im = ax.imshow(feature_map, cmap='viridis')
        ax.set_title(f'Filter {filter_idx+1}\\nResponse', fontsize=8)
        ax.axis('off')
    
    plt.suptitle(f'Image {img_idx+1}: {label} - Original vs Filter Responses', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()